# Independent check of the Methods formulas

This notebook checks that the code computes what the Methods section says. It is fully **read-only**. It reads the saved data and results, and it never changes or overwrites any file.

**How each check works**
1. The Methods formula is shown in words.
2. The formula is computed again here, from scratch, in plain NumPy and SciPy. The project's own functions are **not** used. The aim is to confirm the project code, not to reuse it.
3. The new value is compared with the value the project code saved. **PASS** means they agree within a tiny rounding tolerance.

**How to run**
1. Open Anaconda Prompt and type `conda activate tda`, then `jupyter lab`.
2. Open this notebook and set `ROOT` in the first code cell to your project folder.
3. Click **Run ▸ Run All Cells**. The whole notebook takes about 2 to 5 minutes.

At the end, a summary table lists every check with PASS, FAIL or SKIPPED. SKIPPED means a file was not found.

In [ ]:
# >>> The repository folder. Run this notebook from verification/, or set ROOT to the repository path <<<
import os
ROOT = os.path.abspath("..")

import os, json, zlib, time, warnings
import numpy as np, pandas as pd
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree
warnings.filterwarnings("ignore")
P = lambda *a: os.path.join(ROOT, *a)
assert os.path.isdir(P("results")), "ROOT is not the TDA_paper folder"
PCS = [f"PC{i}" for i in range(1, 11)]
COHORTS = {"GSE41258": "main", "GSE20916": "bulk", "GSE37364": "bulk", "GSE117606": "main"}

RESULTS = []
def check(section, item, mine, stored, tol=1e-6, rel=False):
    mine, stored = float(mine), float(stored)
    diff = abs(mine - stored) / (abs(stored) if rel and stored != 0 else 1)
    ok = diff <= tol
    RESULTS.append(dict(section=section, item=item, recomputed=mine, stored=stored,
                        difference=diff, result="PASS" if ok else "FAIL"))
    print(f"{'PASS' if ok else 'FAIL'}  {item}: recomputed {mine:.6g}, stored {stored:.6g}")
def skipped(section, why):
    RESULTS.append(dict(section=section, item=why, recomputed=np.nan, stored=np.nan, difference=np.nan, result="SKIPPED"))
    print("SKIPPED:", why)
def emb(cohort, version):
    return pd.read_csv(P("results", "step03", "embeddings", f"{cohort}_{COHORTS[cohort]}_{version}_g2000_pc10.csv"), index_col=0)
print("Ready")

## 1. Composition adjustment (Methods, equation 1)

**Methods.** Each gene was regressed on the centred composition scores (epithelial, stromal, immune and, in GSE41258, liver fraction), with an intercept and without stage. The composition part was then removed:

adjusted value = expression − Σ (coefficient × (score − mean score)).

**Project code:** `scripts/step02_composition.py`, function `adjust`.

**Check here.** We solve the least-squares normal equations directly, (XᵀX)b = XᵀY, and compare the result with the saved adjusted file for GSE41258. Differences of about 10⁻⁵ come only from rounding when the CSV file was saved.

In [ ]:
S = "1 Composition adjustment"
try:
    lab = pd.read_csv(P("data/processed/GSE41258_main_samples_qc.csv"), index_col=0)
    g = pd.read_csv(P("data/processed/GSE41258_main_expr_genes.csv"), index_col=0)[lab.index]
    adj_saved = pd.read_csv(P("data/processed/GSE41258_main_expr_genes_adjusted.csv"), index_col=0)[lab.index]
    comp = pd.read_csv(P("results/step02/composition_scores_main.csv"), index_col=0).loc[lab.index]
    C = comp[["Epithelial", "Stromal", "Immune", "Liver_fraction"]].values
    Cc = C - C.mean(axis=0)
    X = np.c_[np.ones(len(Cc)), Cc]
    Y = g.T.values
    b = np.linalg.solve(X.T @ X, X.T @ Y)          # normal equations
    adj_mine = pd.DataFrame((Y - Cc @ b[1:]).T, index=g.index, columns=g.columns)
    common = adj_mine.index.intersection(adj_saved.index)
    d = np.abs(adj_mine.loc[common].values - adj_saved.loc[common].values).max()
    check(S, f"GSE41258 adjusted expression, {len(common)} genes x {g.shape[1]} samples (max difference)", d, 0, tol=1e-4)
except FileNotFoundError as e:
    skipped(S, str(e))

## 2. Composition scores (decoupler univariate linear model)

**Methods.** Each sample was scored with decoupler's univariate linear model on gene-centred log2 expression.

**What that means.** For each sample, the centred expression of all genes is regressed on a 0/1 indicator (1 = marker gene of that compartment). The score is the t-value of the slope.

**Project code:** `scripts/step02_composition.py`, function `ulm_scores` (calls decoupler).

**Check here.** We write the regression and its t-value by hand and compare them with the saved scores.

In [ ]:
S = "2 Composition scores"
try:
    net = pd.read_csv(P("results/step02/marker_sets_used.csv"))
    Xc = g.sub(g.mean(axis=1), axis=0)
    def ulm_t(Xc, w):
        x0 = w - w.mean(); n = len(w)
        Y0 = Xc.values - Xc.values.mean(axis=0)
        slope = x0 @ Y0 / (x0 @ x0)
        res = Y0 - np.outer(x0, slope)
        se = np.sqrt((res ** 2).sum(axis=0) / (n - 2) / (x0 @ x0))
        return slope / se
    for src in ["Epithelial", "Stromal", "Immune"]:
        w = Xc.index.isin(net.loc[net.source == src, "target"]).astype(float)
        d = np.abs(ulm_t(Xc, w) - comp[src].values).max()
        check(S, f"GSE41258 {src} score, all samples (max difference)", d, 0, tol=1e-8)
except (FileNotFoundError, NameError) as e:
    skipped(S, str(e))

## 3. Gene selection and principal component embedding

**Methods.** Remove the 176 liver genes (adjusted GSE41258 only). Remove the lowest 25% of genes by mean expression. Keep the 2,000 genes with the largest variance. Z-score each gene, then take 10 principal components.

**Project code:** `scripts/step03_embedding.py` and `src/tda_crc_pilot.py`, function `preprocess`.

**Check here.** We repeat the gene selection and compute an **exact** PCA by singular value decomposition (SVD). PCA signs are arbitrary, so we compare (a) the gene list, (b) the absolute correlation of each PC and (c) all pairwise distances between samples. The distances are what persistent homology uses. The project PCA used scikit-learn's randomised solver, so differences of about 0.3% are expected and are documented in Supplementary Methods S4.

In [ ]:
S = "3 Embedding"
try:
    liver = pd.read_csv(P("results/step02/liver_signature_main.csv"))["gene"]
    gg = adj_saved.drop(index=[x for x in liver if x in adj_saved.index])
    Xf = gg.dropna(); mu = Xf.mean(axis=1); Xf = Xf[mu > mu.quantile(0.25)]
    genes = list(Xf.var(axis=1).sort_values(ascending=False).index[:2000])
    saved_genes = pd.read_csv(P("results/step03/embeddings/GSE41258_main_adjusted_g2000_pc10_genes.csv"))["gene"]
    check(S, "GSE41258 adjusted: number of selected genes that differ", len(set(genes) ^ set(saved_genes)), 0, tol=0)
    Z = Xf.loc[genes]; Z = Z.sub(Z.mean(axis=1), axis=0).div(Z.std(axis=1), axis=0).T.values
    Z = Z - Z.mean(axis=0)
    U, Sv, Vt = np.linalg.svd(Z, full_matrices=False)
    pcs = U[:, :10] * Sv[:10]
    E = emb("GSE41258", "adjusted").loc[lab.index, PCS].values
    r = min(abs(np.corrcoef(pcs[:, k], E[:, k])[0, 1]) for k in range(10))
    check(S, "GSE41258 adjusted: lowest |correlation| of a PC with exact SVD", r, 1, tol=1e-3)
    rel = np.abs(pdist(pcs) - pdist(E)).max() / np.median(pdist(E))
    check(S, "GSE41258 adjusted: largest pairwise-distance difference / median distance", rel, 0, tol=0.01)
except (FileNotFoundError, NameError) as e:
    skipped(S, str(e))

## 4. Persistent homology: H0 checked against a minimum spanning tree

**Methods.** Vietoris–Rips persistence was computed with GUDHI on Euclidean distances (raw mode). In shape-only mode, the distances were first divided by their median.

**Independent check.** For H0 (connected components), a mathematical fact gives a check that does not use GUDHI. The death times of the H0 features equal the edge lengths of the **minimum spanning tree** of the samples. We compute the tree with SciPy and compare its edges with the saved H0 diagrams. We check subsample 1 of every cohort, data version, mode and stage.

For H1 (loops), no simple formula exists. The H1 results rely on GUDHI, a standard library. They were also cross-checked against a second library, Ripser, in `scripts/crosscheck_D6_ripser.py` (deviation D6).

In [ ]:
S = "4 Persistent homology H0"
try:
    dg = pd.read_csv(P("results/source_data/ph_diagrams_GEO.csv"))
    mem = pd.read_csv(P("results/source_data/ph_subsample_members_GEO.csv"))
    worst, n = 0.0, 0
    for _, r in mem[mem.subsample == 1].iterrows():
        e = emb(r.cohort, r.version)
        D = squareform(pdist(e.loc[r.samples.split(";"), PCS].values))
        if r["mode"] == "shape-only":
            D = D / np.median(D[np.triu_indices_from(D, 1)])
        mst = np.sort(minimum_spanning_tree(D).data)
        h0 = dg[(dg.cohort == r.cohort) & (dg.version == r.version) & (dg["mode"] == r["mode"]) &
                (dg.stage == r.stage) & (dg.subsample == 1) & (dg.dim == "H0")].death.sort_values().values
        worst = max(worst, np.abs(mst - h0).max() if len(mst) == len(h0) else np.inf); n += 1
    check(S, f"H0 deaths = minimum-spanning-tree edges ({n} diagrams, max difference)", worst, 0, tol=1e-9)
except FileNotFoundError as e:
    skipped(S, str(e))

## 5. The 1-Wasserstein distance (Methods, equation 2)

**Methods.** Matching two points costs the larger of their differences in birth and death. An unmatched point is moved to the diagonal at a cost of half its lifetime. W is the smallest total cost.

**Project code:** `src/tda_crc_pilot.py`, function `wasserstein` (Hungarian algorithm).

**Check here.**
(a) A tiny example worked out by hand.
(b) GUDHI's separate Hera algorithm, written by other authors, compared with the project function on real diagrams.

In [ ]:
S = "5 Wasserstein distance"
import gudhi, gudhi.hera
import sys; sys.path.insert(0, P("src"))
from tda_crc_pilot import wasserstein as project_W     # project function, only to be compared
A = np.array([[0.0, 2.0]]); B = np.array([[0.0, 3.0], [1.0, 1.4]])
# by hand: match (0,2)-(0,3): max(0,1)=1 ; leave (1,1.4) unmatched: (1.4-1)/2=0.2 ; total 1.2
check(S, "hand example W = 1.2 (project function)", project_W(A, B), 1.2, tol=1e-12)
check(S, "hand example W = 1.2 (Hera)", gudhi.hera.wasserstein_distance(A, B, order=1, internal_p=np.inf, delta=1e-6), 1.2, tol=1e-9)
try:
    worst = 0
    for (c, v, mo, s1, s2) in [("GSE41258", "adjusted", "raw", "Normal", "Adenoma"),
                               ("GSE41258", "adjusted", "shape-only", "Carcinoma", "Metastasis"),
                               ("GSE20916", "unadjusted", "raw", "Normal", "Carcinoma")]:
        for dim in ("H0", "H1"):
            q = dg[(dg.cohort == c) & (dg.version == v) & (dg["mode"] == mo) & (dg.subsample == 1) & (dg.dim == dim)]
            a = q[q.stage == s1][["birth", "death"]].values; bb = q[q.stage == s2][["birth", "death"]].values
            hw = gudhi.hera.wasserstein_distance(a, bb, order=1, internal_p=np.inf, delta=1e-6)
            worst = max(worst, abs(project_W(a, bb) - hw) / hw)
    check(S, "project W vs Hera on 6 real diagram pairs (largest relative difference)", worst, 0, tol=1e-5)
except NameError as e:
    skipped(S, str(e))

## 6. Test statistic T (Methods, equation 3) recomputed from scratch

**Methods.** Each stage is centred on its own centroid. In shape-only mode, each stage is also divided by its own median pairwise distance. We draw 20 subsamples per stage (m = 80% of the smallest stage), compute diagrams and sum the H0 and H1 Wasserstein distances.

T = mean of the 400 between-stage distances − mean of the 380 within-stage distances.

**Check here.** We rebuild T with our own loop and the Hera distance. We use the same random seed as the project (seed 42, plus a code made from the unit name), so the 20 subsamples are identical. T should therefore match the saved observed T exactly. Tiny differences come from rounding in the saved embedding file. This covers all primary units of the four GEO cohorts, 36 in total. It takes about 1 minute.

In [ ]:
S = "6 Test statistic T"
def diagrams(points, shape):
    D = squareform(pdist(points))
    if shape: D = D / np.median(D[np.triu_indices_from(D, 1)])
    st = gudhi.RipsComplex(distance_matrix=D).create_simplex_tree(max_dimension=2); st.compute_persistence()
    out = []
    for k in (0, 1):
        d = st.persistence_intervals_in_dimension(k)
        out.append(d[np.isfinite(d[:, 1])] if len(d) else np.zeros((0, 2)))
    return out
def Wsum(a, b):
    return sum(gudhi.hera.wasserstein_distance(a[k], b[k], order=1, internal_p=np.inf, delta=1e-6) for k in (0, 1))
def T_from_scratch(E, i_s, i_t, m, shape, rng, k=20):
    E = E.copy()
    for idx in (i_s, i_t):                                   # alignment before permuting
        E[idx] = E[idx] - E[idx].mean(axis=0)
        if shape: E[idx] = E[idx] / np.median(pdist(E[idx]))
    ds = [diagrams(E[rng.choice(i_s, size=m, replace=False)], shape) for _ in range(k)]
    dt = [diagrams(E[rng.choice(i_t, size=m, replace=False)], shape) for _ in range(k)]
    between = [Wsum(a, b) for a in ds for b in dt]
    within = [Wsum(ds[a], ds[b]) for a in range(k) for b in range(a + 1, k)] + \
             [Wsum(dt[a], dt[b]) for a in range(k) for b in range(a + 1, k)]
    return np.mean(between) - np.mean(within)

UNITS = []
for c in COHORTS:
    stages = ["Normal", "Adenoma", "Carcinoma"] + (["Metastasis"] if c == "GSE41258" else [])
    for v in ("unadjusted", "adjusted"):
        for s, t in zip(stages[:-1], stages[1:]):
            for mode in ("shape", "raw"):
                if c == "GSE41258":
                    name = f"primary__{'unadj' if v == 'unadjusted' else 'adj'}__{s[0]}-{t[0]}__{mode}"
                    path = P("results/step05/checkpoints", name + ".json")
                else:
                    name = f"{c}__primary__{v[:5]}__{s[0]}-{t[0]}__{mode}"
                    path = P("results/step05", c, "checkpoints", name + ".json")
                UNITS.append((c, v, s, t, mode, name, path, stages))
t0 = time.time()
for c, v, s, t, mode, name, path, stages in UNITS:
    if not os.path.exists(path):
        skipped(S, f"{name}: checkpoint not found"); continue
    rec = json.load(open(path))
    e = emb(c, v); E = e[PCS].values; st = e.stage.values
    m = int(0.8 * min((st == x).sum() for x in stages))
    i_s, i_t = np.where(st == s)[0], np.where(st == t)[0]
    if c == "GSE41258": m = min(m, int(0.8 * min(len(i_s), len(i_t))))
    rng = np.random.default_rng(np.random.SeedSequence(42, spawn_key=(zlib.crc32(name.encode()),)))
    check(S, f"{name} T", T_from_scratch(E, i_s, i_t, m, mode == "shape", rng), rec["T_obs"], tol=1e-4, rel=True)
print(f"{time.time() - t0:.0f} s")

## 7. Permutation p-value, z-score and Holm correction

**Methods (Supplementary Methods S11).**
- p = (1 + number of permuted T ≥ observed T) / (999 + 1).
- z = (observed T − mean of the null) / SD of the null.
- Holm: sort the K p-values. Multiply the i-th smallest by (K − i + 1). Then take the running maximum, capped at 1.

**Check here.** Each saved checkpoint holds all 999 permuted T values. We recompute p and z from them, and the Holm values from the p-values, with our own code and with statsmodels.

In [ ]:
S = "7 p-value, z, Holm"
from statsmodels.stats.multitest import multipletests
def holm(p):
    p = np.asarray(p, float); K = len(p); out = np.empty(K); run = 0
    for i, j in enumerate(np.argsort(p)):
        run = max(run, (K - i) * p[j]); out[j] = min(1, run)
    return out
for c, v, s, t, mode, name, path, stages in UNITS:
    if not os.path.exists(path): continue
    rec = json.load(open(path)); null = np.array(rec["null"])
    check(S, f"{name} p", (1 + np.sum(null >= rec["T_obs"])) / (1 + len(null)), rec["p"], tol=1e-12)
    check(S, f"{name} z", (rec["T_obs"] - null.mean()) / null.std(ddof=1), rec["z"], tol=1e-6, rel=True)
for c in COHORTS:
    f = P("results/step05/permutation_results_all.csv") if c == "GSE41258" else P("results/step05", c, "permutation_results_all.csv")
    if not os.path.exists(f): skipped(S, f"{c} permutation_results_all.csv not found"); continue
    pr = pd.read_csv(f); pr = pr[pr.analysis == "primary"]
    for (v, mo), gq in pr.groupby(["version", "mode"]):
        check(S, f"{c} {v} {mo}: Holm (own code, max difference)", np.abs(holm(gq.p.values) - gq.p_holm.values).max(), 0, tol=1e-12)
        check(S, f"{c} {v} {mo}: Holm (statsmodels, max difference)",
              np.abs(multipletests(gq.p.values, method="holm")[1] - gq.p_holm.values).max(), 0, tol=1e-12)

### 7b. Optional: a fresh permutation test with new random draws

This repeats one test with **new** random permutations (99 instead of 999, to save time). The p-value will not be identical, because the draws are different, but it should be in the same range. Set `RUN_SLOW = True` to run it. It takes about 1 to 2 minutes.

In [ ]:
RUN_SLOW = False
if RUN_SLOW:
    c, v, s, t, mode, name, path, stages = UNITS[0]
    e = emb(c, v); E = e[PCS].values; st = e.stage.values
    i_s, i_t = np.where(st == s)[0], np.where(st == t)[0]
    m = min(int(0.8 * min((st == x).sum() for x in stages)), int(0.8 * min(len(i_s), len(i_t))))
    rng = np.random.default_rng(2026)
    Tobs = T_from_scratch(E, i_s, i_t, m, mode == "shape", rng)
    pool = np.r_[i_s, i_t]; null = []
    for _ in range(99):
        q = rng.permutation(pool)
        null.append(T_from_scratch(E, q[:len(i_s)], q[len(i_s):], m, mode == "shape", rng))
    print(name, "fresh p =", (1 + np.sum(np.array(null) >= Tobs)) / 100, " saved p =", json.load(open(path))["p"])

## 8. Random-effects meta-analysis (DerSimonian–Laird)

**Methods.** The cohort z-scores were pooled with within-cohort variance 1. With k cohorts:
- Q = Σ (z − mean z)².
- τ² = max(0, (Q − (k − 1)) / (k − 1)). In the textbook DerSimonian–Laird method, τ² is never below 0.
- pooled z = mean z, and SE = √((1 + τ²) / k).
- 95% CI = pooled z ± 1.96 SE, with a two-sided normal p-value.
- I² = max(0, (Q − (k − 1)) / Q).

**Project code:** `scripts/meta_analysis_NA.py` and `scripts/meta_analysis_4cohort.py` (statsmodels `combine_effects`).

**Known finding (expected FAIL).** For adenoma → carcinoma in adjusted data (four cohorts), Q (2.12) is smaller than k − 1 (3). The textbook method then sets τ² = 0. statsmodels 0.15.0 did not do this. It used a negative τ² (−0.29), which made the confidence interval slightly too narrow.
- Correct values: pooled z = 0.25, 95% CI −0.73 to 1.23, p = 0.62.
- Values in the paper: CI −0.58 to 1.07, p = 0.55.

The pooled z and the conclusion (not significant) do not change. No other row is affected.

In [ ]:
S = "8 Meta-analysis"
for folder, by in [("meta_analysis", ["version"]), ("meta_analysis_4cohort", ["transition", "version"])]:
    try:
        mi = pd.read_csv(P("results", folder, "meta_inputs.csv")); mr = pd.read_csv(P("results", folder, "meta_results.csv"))
    except FileNotFoundError as e:
        skipped(S, str(e)); continue
    for _, r in mr.iterrows():
        sel = np.ones(len(mi), bool)
        for col in by: sel &= (mi[col] == r[col]).values
        z = mi[sel].z.values; k = len(z)
        Q = ((z - z.mean()) ** 2).sum(); tau2 = max(0.0, (Q - (k - 1)) / (k - 1))
        se = np.sqrt((1 + tau2) / k); pz = z.mean()
        lab_ = f"{folder} {' '.join(str(r[col]) for col in by)}"
        check(S, f"{lab_}: pooled z", pz, r.pooled_z, tol=1e-6)
        check(S, f"{lab_}: CI lower", pz - 1.959964 * se, r.ci_low, tol=1e-4)
        check(S, f"{lab_}: two-sided p", 2 * stats.norm.sf(abs(pz / se)), r.p_two_sided, tol=1e-4)

## 9. Between-sample dispersion (PERMDISP) and the dispersion ratio D

**Methods.**
- Distance of each sample to the centroid (mean position) of its own stage.
- D = mean distance in the later stage ÷ mean distance in the earlier stage.
- Test: one-way ANOVA F on the distances.
- Bias-adjusted distance = distance × √(n / (n − 1)).

**Project code:** `scripts/step13_benchmark.py` (`permdisp`) and `scripts/addendum9_heterogeneity.py` (`loo_dist`).

**Check here.** We recompute every distance, D and F, using SciPy's `f_oneway` for F, for all four GEO cohorts in both data versions. The p-value is also recomputed with 999 **new** permutations. It will not be identical, but it should be close.

In [ ]:
S = "9 PERMDISP"
def cdist_(X, bias=False):
    n = len(X); d = np.linalg.norm(X - X.mean(axis=0), axis=1)
    return d * np.sqrt(n / (n - 1)) if bias else d
try:
    sd = pd.read_csv(P("results/source_data/stage_centroid_distances_GEO.csv"))
    bm = pd.read_csv(P("results/step13/benchmark_results.csv"))
    rng = np.random.default_rng(7)
    for c in COHORTS:
        for v in ("unadjusted", "adjusted"):
            e = emb(c, v); s_ = sd[(sd.cohort == c) & (sd.version == v)].set_index("sample")
            w1 = w2 = 0
            for stg in e.stage.unique():
                Xs = e[e.stage == stg]
                w1 = max(w1, np.abs(cdist_(Xs[PCS].values) - s_.loc[Xs.index, "dist_to_centroid"].values).max())
                w2 = max(w2, np.abs(cdist_(Xs[PCS].values, True) - s_.loc[Xs.index, "dist_to_centroid_bias_adj"].values).max())
            check(S, f"{c} {v}: distances to centroid (max difference)", w1, 0, tol=1e-9)
            check(S, f"{c} {v}: bias-adjusted distances (max difference)", w2, 0, tol=1e-9)
            for _, r in bm[(bm.cohort == c) & (bm.version == v)].iterrows():
                a, b = r.transition.split("->")
                dA, dB = cdist_(e[e.stage == a][PCS].values), cdist_(e[e.stage == b][PCS].values)
                check(S, f"{c} {v} {r.transition}: D", dB.mean() / dA.mean(), r.permdisp_effect, tol=1e-9)
                F = stats.f_oneway(dA, dB).statistic
                check(S, f"{c} {v} {r.transition}: F", F, r.permdisp_stat, tol=1e-6, rel=True)
                zz = np.r_[dA, dB]; n1 = len(dA)
                null = [stats.f_oneway(*np.split(rng.permutation(zz), [n1])).statistic for _ in range(999)]
                pnew = (1 + np.sum(np.array(null) >= F)) / 1000
                check(S, f"{c} {v} {r.transition}: p with new permutations (|difference| < 0.05)", pnew, r.permdisp_p, tol=0.05)
except FileNotFoundError as e:
    skipped(S, str(e))

## 10. Share of excess dispersion explained by CMS (Methods, equation 4)

**Methods.**
- D_total = carcinoma ÷ adenoma dispersion, using bias-adjusted distances to the centroid of all carcinomas.
- D_within = the same ratio, using each carcinoma's distance to the centroid of its own subtype. Subtypes with fewer than 3 carcinomas are left out.
- f = (D_total − D_within) / (D_total − 1).

**Project code:** `scripts/addendum9_heterogeneity.py`, function `decompose`.

In [ ]:
S = "10 CMS share f"
try:
    cms = pd.read_csv(P("results/addendum9/cms_labels.csv")).set_index("sample")
    dd = pd.read_csv(P("results/addendum9/dispersion_decomposition.csv"))
    for c in COHORTS:
        r = dd[(dd.cohort == c) & (dd.labels == "primary")].iloc[0]
        e = emb(c, "adjusted"); A = e[e.stage == "Adenoma"][PCS].values
        Ce = e[e.stage == "Carcinoma"]; Cm = Ce[PCS].values; lab_c = cms.loc[Ce.index, "nearest"].values
        dA = cdist_(A, True); Dt = cdist_(Cm, True).mean() / dA.mean()
        keep = [k for k in np.unique(lab_c) if (lab_c == k).sum() >= 3]
        Dw = np.concatenate([cdist_(Cm[lab_c == k], True) for k in keep]).mean() / dA.mean()
        check(S, f"{c}: D_total", Dt, r.D_total, tol=1e-9)
        check(S, f"{c}: D_within", Dw, r.D_within, tol=1e-9)
        check(S, f"{c}: f", (Dt - Dw) / (Dt - 1), r.f, tol=1e-8)
except FileNotFoundError as e:
    skipped(S, str(e))

## 11. Share of excess dispersion explained by chromosome-arm scores

**Methods.** Within each stage, every principal component is regressed on the arm score (20q alone, or 20q, 20p, 13q and 8q together). The residuals are kept. D_resid is the carcinoma ÷ adenoma dispersion of the residuals, and f = (D_total − D_resid) / (D_total − 1).

**Project code:** `scripts/addendum11_differentiation.py`, functions `resid_within` and `removal`.

In [ ]:
S = "11 Arm-score share f"
def resid(Xs, s):
    Xd = np.c_[np.ones(len(s)), s]; return Xs - Xd @ np.linalg.lstsq(Xd, Xs, rcond=None)[0]
try:
    arm = pd.read_csv(P("results/hgnc/source_data/arm_scores_GEO.csv"))
    b3 = pd.read_csv(P("results/hgnc/addendum11/partB_differentiation/B3_dispersion_decomposition.csv")).set_index("cohort")
    for c in COHORTS:
        a_ = arm[(arm.cohort == c) & (arm.version == "adjusted") & (arm.genes == "excl122")].set_index("sample")
        e = emb(c, "adjusted"); iA = e.index[e.stage == "Adenoma"]; iC = e.index[e.stage == "Carcinoma"]
        Dt = cdist_(e.loc[iC, PCS].values, True).mean() / cdist_(e.loc[iA, PCS].values, True).mean()
        for cols, key in [(["arm_20q"], "chr20q"), (["arm_20q", "arm_20p", "arm_13q", "arm_8q"], "chr4")]:
            RA = resid(e.loc[iA, PCS].values, a_.loc[iA, cols].values)
            RC = resid(e.loc[iC, PCS].values, a_.loc[iC, cols].values)
            Dr = cdist_(RC, True).mean() / cdist_(RA, True).mean()
            check(S, f"{c} {key}: D_resid", Dr, b3.loc[c, f"{key}_D_resid"], tol=1e-8)
            check(S, f"{c} {key}: f", (Dt - Dr) / (Dt - 1), b3.loc[c, f"{key}_f"], tol=1e-6)
except FileNotFoundError as e:
    skipped(S, str(e))

## 12. Variability ratio V and Brown–Forsythe test

**Methods.**
- V = median absolute deviation (MAD) in carcinoma ÷ MAD in adenoma.
- Brown–Forsythe test: a one-way ANOVA on each sample's absolute distance from its own group median.

**Project code:** `scripts/addendum9_heterogeneity.py`, function `variability`.

**Check here.** We compute both by hand for every tested gene in GSE41258 and compare them with the saved table. The saved table uses current HGNC symbols, so only genes with the same symbol in both files are compared.

In [ ]:
S = "12 Variability ratio and Brown-Forsythe"
try:
    var = pd.read_csv(P("results/hgnc/addendum9/variability_GSE41258.csv"), index_col=0)
    Xv = adj_saved.dropna(); mu = Xv.mean(axis=1); Xv = Xv[mu > mu.quantile(0.25)]
    common = Xv.index.intersection(var.index)
    A = Xv.loc[common, lab.stage.values == "Adenoma"].values; Cc_ = Xv.loc[common, lab.stage.values == "Carcinoma"].values
    madA = np.median(np.abs(A - np.median(A, axis=1, keepdims=True)), axis=1)
    madC = np.median(np.abs(Cc_ - np.median(Cc_, axis=1, keepdims=True)), axis=1)
    V = madC / madA
    check(S, f"GSE41258 V, {len(common)} genes (max difference)", np.nanmax(np.abs(V - var.loc[common, "V"].values)), 0, tol=1e-9)
    za = np.abs(A - np.median(A, axis=1, keepdims=True)); zc = np.abs(Cc_ - np.median(Cc_, axis=1, keepdims=True))
    pbf = stats.f_oneway(zc, za, axis=1).pvalue
    check(S, f"GSE41258 Brown-Forsythe p, {len(common)} genes (max difference)", np.nanmax(np.abs(pbf - var.loc[common, "p"].values)), 0, tol=1e-9)
except (FileNotFoundError, NameError) as e:
    skipped(S, str(e))

## 13. Progression score, valley ratio and AUC (gradual versus switch-like change)

**Methods.**
- Score = (PC1 − normal median) / (carcinoma median − normal median).
- Valley ratio = number of samples in the middle third between the two stage medians ÷ the mean number in windows of the same width centred on each median.
- AUC = probability that a sample of the later stage scores higher than a sample of the earlier stage.

**Project code:** `scripts/addendum8_gradual_switch.py`, functions `scale_score`, `valley_ratio` and `continuity`.

In [ ]:
S = "13 Progression score, VR, AUC"
try:
    ps = pd.read_csv(P("results/addendum8/progression_scores.csv"))
    cr = pd.read_csv(P("results/addendum8/continuity_results.csv")).set_index("cohort")
    def VR(x, y):
        mx, my = np.median(x), np.median(y); w = my - mx; a = np.r_[x, y]
        mid = ((a >= mx + w / 3) & (a <= mx + 2 * w / 3)).sum()
        rx = ((a >= mx - w / 6) & (a <= mx + w / 6)).sum(); ry = ((a >= my - w / 6) & (a <= my + w / 6)).sum()
        return mid / ((rx + ry) / 2)
    def AUC(x, y): return (np.greater.outer(y, x).sum() + 0.5 * np.equal.outer(y, x).sum()) / (len(x) * len(y))
    for c in COHORTS:
        p_ = ps[ps.cohort == c].set_index("sample"); e = emb(c, "adjusted").loc[p_.index]
        st = e.stage.values; pc1 = e.PC1.values
        s = (pc1 - np.median(pc1[st == "Normal"])) / (np.median(pc1[st == "Carcinoma"]) - np.median(pc1[st == "Normal"]))
        check(S, f"{c}: progression score (max difference)", np.abs(s - p_.s_allgene.values).max(), 0, tol=1e-9)
        check(S, f"{c}: adenoma median a*", np.median(s[st == "Adenoma"]), cr.loc[c, "astar"], tol=1e-9)
        for tr, (x, y) in {"NA": ("Normal", "Adenoma"), "AC": ("Adenoma", "Carcinoma")}.items():
            check(S, f"{c} {tr}: AUC", AUC(s[st == x], s[st == y]), cr.loc[c, f"{tr}_AUC"], tol=1e-9)
            if pd.notna(cr.loc[c, f"{tr}_VR"]):
                check(S, f"{c} {tr}: valley ratio", VR(s[st == x], s[st == y]), cr.loc[c, f"{tr}_VR"], tol=1e-9)
except FileNotFoundError as e:
    skipped(S, str(e))

## Summary

The table lists every check. The second table shows only the checks that did not pass, if any. A FAIL does not automatically mean the paper is wrong. It means that number needs a closer look, so please send it to Claude to examine.

In [ ]:
res = pd.DataFrame(RESULTS)
print(res.groupby(["section", "result"]).size().unstack(fill_value=0))
print("\nTOTAL:", res.result.value_counts().to_dict())
pd.set_option("display.max_colwidth", 120)
res[res.result != "PASS"]